# HW2: Transformers

## Setup Instructions

1. **Make a personal copy of this notebook.** You will not be able to modify the master version of this assignment.
1. **Setup your GPU runtime**: Go to Runtime > Change Runtime Type and select "T4 GPU". Click "Save".
1. **Upload the Pile data**: On the "Files" tab, upload the `pile` data folder. You can also upload the `pile.zip` directly and use the provided cell to unzip things into the right place. The `pile/` folder should contain train.txt, dev.txt, and test.txt files. The code expects this folder to be in `content/pile/`, where `/content/` is the default home folder for Colab projects.

## TODOs

- You must first implement the `Transformer` and `MultiHeadTransformer` classes. After you've completed these classes and finished the notebook, you will copy and paste this cell into a Python script called `transformer.py`, and upload this to the Gradescope autograder.
- You must later implement the `generate` function; this should be an implementation of temperature sampling.
- After you've completed the full notebook, you will upload both `transformer.py` and your completed notebook (an `.ipynb` file) to Gradescope.

# Environment Setup

These cells download the dependencies you'll need, and also set up any needed environment variables.

In [1]:
# Verify installation (no need to pip install locally)
import torch
import torch.nn.functional as F
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
elif torch.backends.mps.is_available():
    print("Using Apple Silicon MPS")

PyTorch version: 2.7.0
CUDA available: False
MPS available: True
Using Apple Silicon MPS


In [2]:
# Set the pile data path for local running
import os

PILE_DATA_ROOT = '/Users/tangzhengzheng/Desktop/BU/505/hw2/pile/'

# Data Loading Utilities

These functions load data from the Pile or from a small Shakespeare corpus (from HuggingFace).

In [3]:
import torch
from pathlib import Path
from transformers import AutoTokenizer
from datasets import load_dataset

def get_tokenizer():
    # We'll use the GPT-2 tokenizer.
    _tokenizer = AutoTokenizer.from_pretrained("gpt2")
    return _tokenizer

def load_pile_data(split, max_len=100):
    print(f"Loading Pile {split} data...")

    file_path = Path(PILE_DATA_ROOT) / f"{split}.txt"
    if not file_path.exists():
        raise FileNotFoundError(f"Cannot find file: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    tokenizer = get_tokenizer()
    rows = []
    for line in lines:
        ids = tokenizer.encode(line)[:max_len]
        if len(ids) < max_len:
            ids += [tokenizer.eos_token_id] * (max_len - len(ids))
        rows.append(ids)

    return torch.tensor(rows, dtype=torch.long)

def load_shakespeare_data(split='train', max_len=100, max_samples=None):
    print(f"Loading Shakespeare {split} data from HuggingFace...")
    dataset = load_dataset("2nji/Shakespeare_Corpus", split=split)

    # Tokenize
    tokenizer = get_tokenizer()

    # Extract text and tokenize
    rows = []
    for i, item in enumerate(dataset):
        if max_samples and i >= max_samples:
            break

        # Get text from the dataset (adjust key if needed - might be 'text', 'content', etc.)
        text = item.get('text', '') or item.get('content', '') or str(item)

        # Tokenize and truncate to max_len
        ids = tokenizer.encode(text)[:max_len]
        if len(ids) < max_len:
            ids += [tokenizer.eos_token_id] * (max_len - len(ids))
        rows.append(ids)

    print(f"Loaded {len(rows)} Shakespeare {split} tokens")
    return torch.tensor(rows, dtype=torch.long)

def load_shakespeare_data_hf(split='train', max_samples=None):
    print(f"Loading Shakespeare {split} data for HuggingFace...")

    # Load the dataset with the correct split
    dataset = load_dataset("2nji/Shakespeare_Corpus", split=split)

    # Extract text from each item in the dataset
    rows = []
    for i, item in enumerate(dataset):
        if max_samples and i >= max_samples:
            break

        # Handle different possible text field names
        text = item.get('text', '') or item.get('content', '') or str(item)
        # Split into lines if needed and filter empty lines
        rows.append(text)

    print(f"Loaded {len(rows)} Shakespeare {split} lines")
    return [{"text": row} for row in rows]

# Transformer Implementation

This is your main TODO. You will implement the `Transformer` and `MultiHeadTransformer` classes in the following cell.

You can try implementing `Transformer` first and testing it in the following cells before coming back to implement `MultiHeadTransformer`.

After you've completed the notebook and answered all questions in the written report, copy the contents of this cell and paste it into a script called `transformer.py`. Upload this script, as well as your .ipynb notebook, to HW2's code portal on Gradescope. (You can download your notebook by clicking File > Download > Download .ipynb).

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class Transformer(nn.Module):
    """Multi-layer single-head Transformer decoder."""

    def __init__(self, vocab_size, hidden_dim, context_len, num_layers=2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.context_len = context_len
        self.num_layers = num_layers

        # 1. Token embeddings
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        # Positional embeddings
        self.pos_embedding = nn.Embedding(context_len, hidden_dim)

        # Create layers dynamically
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            # STUDENT START -----------------------------------------
            layer = nn.ModuleDict({
                'W_Q': nn.Linear(hidden_dim, hidden_dim),
                'W_K': nn.Linear(hidden_dim, hidden_dim),
                'W_V': nn.Linear(hidden_dim, hidden_dim),
                'W_O': nn.Linear(hidden_dim, hidden_dim),
                'W_up': nn.Linear(hidden_dim, 4 * hidden_dim),
                'W_down': nn.Linear(4 * hidden_dim, hidden_dim),
            })
            self.layers.append(layer)
            # STUDENT END --------------------------------------------

        # Layer norm parameters for each layer
        self.gamma_attn = nn.ParameterList([nn.Parameter(torch.ones(hidden_dim)) for _ in range(num_layers)])
        self.beta_attn = nn.ParameterList([nn.Parameter(torch.zeros(hidden_dim)) for _ in range(num_layers)])
        self.gamma_mlp = nn.ParameterList([nn.Parameter(torch.ones(hidden_dim)) for _ in range(num_layers)])
        self.beta_mlp = nn.ParameterList([nn.Parameter(torch.zeros(hidden_dim)) for _ in range(num_layers)])

    def layer_norm(self, x, gamma, beta):
        # STUDENT START ------------------------------
        mu = x.mean(dim=-1, keepdim=True)
        sigma = x.std(dim=-1, keepdim=True, unbiased=False)
        x_hat = gamma * (x - mu) / (sigma + 1e-5) + beta
        return x_hat
        # STUDENT END --------------------------------

    def forward(self, x):
        B, T = x.size()  # (Batch size, sequence length)

        # 1. Token embeddings + positional encodings
        positions = torch.arange(0, T, device=x.device).unsqueeze(0)
        h = self.embedding(x) + self.pos_embedding(positions)

        # Process through each layer
        for layer_idx in range(self.num_layers):
            layer = self.layers[layer_idx]
            residual = h

            # STUDENT START --------------------------
            # i. Project X into Q, K, and V matrices
            Q = layer['W_Q'](h)  # (B, T, hidden_dim)
            K = layer['W_K'](h)  # (B, T, hidden_dim)
            V = layer['W_V'](h)  # (B, T, hidden_dim)

            # ii. Compute attention scores
            d_k = K.size(-1)
            attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (B, T, T)

            # iii. Causal masking
            causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
            attn_scores = attn_scores.masked_fill(causal_mask, float('-inf'))

            # iv. Softmax and multiply by values
            attn_weights = F.softmax(attn_scores, dim=-1)  # (B, T, T)
            attn_output = torch.matmul(attn_weights, V)  # (B, T, hidden_dim)

            # v. Output projection
            attn_output = layer['W_O'](attn_output)  # (B, T, hidden_dim)

            # vi. Residual and LayerNorm
            h = self.layer_norm(residual + attn_output, self.gamma_attn[layer_idx], self.beta_attn[layer_idx])
            # STUDENT END ------------------------------

            # MLP
            # STUDENT START -------------------------
            residual = h
            mlp_output = layer['W_up'](h)  # (B, T, 4 * hidden_dim)
            mlp_output = F.relu(mlp_output)
            mlp_output = layer['W_down'](mlp_output)  # (B, T, hidden_dim)
            # STUDENT END ----------------------------

            # Residual & layer norm
            # STUDENT START ----------------------------
            h = self.layer_norm(residual + mlp_output, self.gamma_mlp[layer_idx], self.beta_mlp[layer_idx])
            # STUDENT END ------------------------------

        return h


class MultiHeadTransformer(Transformer):
    """Multi-layer multi-head Transformer decoder."""

    def __init__(self, vocab_size, hidden_dim, context_len, num_heads=4, num_layers=2):
        super().__init__(vocab_size, hidden_dim, context_len, num_layers)
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        assert hidden_dim % num_heads == 0, "Hidden dim must be divisible by num_heads"

    def forward(self, x):
        B, T = x.size()

        # Embeddings
        positions = torch.arange(0, T, device=x.device).unsqueeze(0)
        h = self.embedding(x) + self.pos_embedding(positions)

        # Process through each layer
        for layer_idx in range(self.num_layers):
            layer = self.layers[layer_idx]
            residual = h

            # STUDENT START ------------------------------------
            # i. Project X into Q, K, and V matrices
            Q = layer['W_Q'](h)  # (B, T, hidden_dim)
            K = layer['W_K'](h)  # (B, T, hidden_dim)
            V = layer['W_V'](h)  # (B, T, hidden_dim)

            # ii. Reshape for multi-head: (B, T, hidden_dim) -> (B, num_heads, T, head_dim)
            Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
            K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
            V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

            # iii. Compute attention scores
            attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

            # iv. Causal masking
            causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
            attn_scores = attn_scores.masked_fill(causal_mask, float('-inf'))

            # v. Softmax and multiply by values
            attn_weights = F.softmax(attn_scores, dim=-1)
            attn_output = torch.matmul(attn_weights, V)  # (B, num_heads, T, head_dim)

            # vi. Concatenate heads
            attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, self.hidden_dim)

            # vii. Output projection
            attn_output = layer['W_O'](attn_output)

            # viii. Residual and LayerNorm
            h = self.layer_norm(residual + attn_output, self.gamma_attn[layer_idx], self.beta_attn[layer_idx])

            # ix. MLP
            residual = h
            mlp_output = layer['W_up'](h)
            mlp_output = F.relu(mlp_output)
            mlp_output = layer['W_down'](mlp_output)

            # x. Residual and LayerNorm
            h = self.layer_norm(residual + mlp_output, self.gamma_mlp[layer_idx], self.beta_mlp[layer_idx])
            # STUDENT END --------------------------------------------------------

        return h

# Training and Evaluation Utilities

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

def _ensure_2d_batch(batch):
    """Ensure batch is 2D tensor."""
    if isinstance(batch, (list, tuple)):
        if len(batch) == 1 and torch.is_tensor(batch[0]):
            batch = batch[0]
        else:
            batch = torch.stack([b if torch.is_tensor(b) else torch.tensor(b) for b in batch], dim=0)

    if not torch.is_tensor(batch):
        batch = torch.tensor(batch)

    if batch.dim() == 1:
        batch = batch.unsqueeze(0)

    return batch

def train(model, train_data, dev_data, epochs=10, lr=5e-3, device='cpu', save_path='model.pt'):
    """Train a transformer model."""
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    model.to(device)
    model.train()

    print(f"Starting training on {device}...")

    train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
    dev_loader = DataLoader(dev_data, batch_size=16)

    best_perp = float('inf')
    prev_perp = float('inf')

    for epoch in range(epochs):
        total_loss = 0
        for i, batch in enumerate(train_loader):
            batch = _ensure_2d_batch(batch)
            if batch.size(1) < 2:
                continue

            inputs = batch[:, :-1].to(device)
            targets = batch[:, 1:].to(device)

            optimizer.zero_grad()
            output = model(inputs)
            logits = torch.matmul(output, model.embedding.weight.t())
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            if i % 100 == 0:
                print(f"Epoch {epoch} Step {i} Loss: {loss.item():.4f}")

        val_perp = evaluate_perplexity(model, dev_loader, device)
        print(f"Epoch {epoch} Completed. Dev Perplexity: {val_perp:.4f}")

        if val_perp > prev_perp * 1.1 and epoch > 2:
            print(f"Early stopping at epoch {epoch} due to increasing perplexity.")
            break

        prev_perp = val_perp
        if val_perp < best_perp:
            best_perp = val_perp
            torch.save(model.state_dict(), save_path)
            print(f"Best model saved to {save_path} with perplexity {best_perp:.4f}")

    if best_perp == float('inf'):
        torch.save(model.state_dict(), save_path)
        print(f"Model saved to {save_path}")


def evaluate_perplexity(model, data, device='cpu'):
    """Evaluate perplexity on dataset."""
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0
    total_tokens = 0

    with torch.no_grad():
        for batch in data:
            batch = _ensure_2d_batch(batch)
            if batch.size(1) < 2:
                continue

            inputs = batch[:, :-1].to(device)
            targets = batch[:, 1:].to(device)

            output = model(inputs)
            logits = torch.matmul(output, model.embedding.weight.t())
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            total_loss += loss.item() * targets.numel()
            total_tokens += targets.numel()

    model.train()

    if total_tokens == 0:
        return float("inf")

    avg_loss = total_loss / total_tokens
    perplexity = torch.exp(torch.tensor(avg_loss))
    return perplexity.item()

# Training from Scratch on the Pile

Train your transformer on general internet text (The Pile).

In [6]:
# Load The Pile
print("Loading The Pile...")
train_data = load_pile_data('train')
dev_data = load_pile_data('dev')
tokenizer = get_tokenizer()

print(f"Train data shape: {train_data.shape}")
print(f"Dev data shape: {dev_data.shape}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Use GPU if available (CUDA or MPS for Apple Silicon)
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

Loading The Pile...
Loading Pile train data...


Token indices sequence length is longer than the specified maximum sequence length for this model (1895 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2018 > 1024). Running this sequence through the model will result in indexing errors


Loading Pile dev data...
Train data shape: torch.Size([10000, 100])
Dev data shape: torch.Size([1000, 100])
Vocabulary size: 50257
Using device: mps


In [7]:
# Initialize single-head transformer
vocab_size = tokenizer.vocab_size
hidden_dim = 256
context_len = 128

model_single = Transformer(vocab_size=vocab_size, hidden_dim=hidden_dim, context_len=context_len)
print(f"Model parameters: {sum(p.numel() for p in model_single.parameters()):,}")

# Train model. These are typical pre-training hyperparameters; you can feel
# free to modify the number of epochs and learning rate.
train(
    model_single,
    train_data,
    dev_data,
    epochs=10,  # Adjust as needed
    lr=5e-3,
    device=device,
    save_path='TRANSFORMER_pile.pt'
)

# Evaluate
perplexity = evaluate_perplexity(model_single, dev_data, device)
print(f"\nFinal Pile Dev Perplexity: {perplexity:.4f}")

Model parameters: 14,478,080
Starting training on mps...
Epoch 0 Step 0 Loss: 158.6577
Epoch 0 Step 100 Loss: 9.8408
Epoch 0 Step 200 Loss: 8.7772
Epoch 0 Step 300 Loss: 8.2077
Epoch 0 Step 400 Loss: 8.4147
Epoch 0 Step 500 Loss: 8.2728
Epoch 0 Step 600 Loss: 8.4010
Epoch 0 Completed. Dev Perplexity: 3199.3730
Best model saved to TRANSFORMER_pile.pt with perplexity 3199.3730
Epoch 1 Step 0 Loss: 8.0867
Epoch 1 Step 100 Loss: 8.1924
Epoch 1 Step 200 Loss: 7.8181
Epoch 1 Step 300 Loss: 8.0706
Epoch 1 Step 400 Loss: 8.0984
Epoch 1 Step 500 Loss: 8.0461
Epoch 1 Step 600 Loss: 8.1074
Epoch 1 Completed. Dev Perplexity: 2905.7791
Best model saved to TRANSFORMER_pile.pt with perplexity 2905.7791
Epoch 2 Step 0 Loss: 7.8769
Epoch 2 Step 100 Loss: 7.9573
Epoch 2 Step 200 Loss: 8.3020
Epoch 2 Step 300 Loss: 7.8562
Epoch 2 Step 400 Loss: 7.9780
Epoch 2 Step 500 Loss: 8.1820
Epoch 2 Step 600 Loss: 7.8833
Epoch 2 Completed. Dev Perplexity: 2869.6260
Best model saved to TRANSFORMER_pile.pt with perpl

In [8]:
# Initialize multi-head transformer
model_pile = MultiHeadTransformer(
    vocab_size=vocab_size,
    hidden_dim=hidden_dim,
    context_len=context_len,
    num_heads=4
)
print(f"Model parameters: {sum(p.numel() for p in model_pile.parameters()):,}")

# Train model
train(
    model_pile,
    train_data,
    dev_data,
    epochs=10,  # Adjust as needed
    lr=5e-3,
    device=device,
    save_path='TRANSFORMER_MH_pile.pt'
)

# Evaluate
perplexity = evaluate_perplexity(model_pile, dev_data, device)
print(f"\nFinal Pile Dev Perplexity: {perplexity:.4f}")

Model parameters: 14,478,080
Starting training on mps...
Epoch 0 Step 0 Loss: 168.3683
Epoch 0 Step 100 Loss: 10.0909
Epoch 0 Step 200 Loss: 8.9199
Epoch 0 Step 300 Loss: 8.4272
Epoch 0 Step 400 Loss: 7.6254
Epoch 0 Step 500 Loss: 7.7453
Epoch 0 Step 600 Loss: 7.9544
Epoch 0 Completed. Dev Perplexity: 2764.0015
Best model saved to TRANSFORMER_MH_pile.pt with perplexity 2764.0015
Epoch 1 Step 0 Loss: 7.6991
Epoch 1 Step 100 Loss: 7.9748
Epoch 1 Step 200 Loss: 7.5412
Epoch 1 Step 300 Loss: 8.2140
Epoch 1 Step 400 Loss: 7.7765
Epoch 1 Step 500 Loss: 7.7235
Epoch 1 Step 600 Loss: 8.0049
Epoch 1 Completed. Dev Perplexity: 2426.9709
Best model saved to TRANSFORMER_MH_pile.pt with perplexity 2426.9709
Epoch 2 Step 0 Loss: 7.9802
Epoch 2 Step 100 Loss: 7.7844
Epoch 2 Step 200 Loss: 8.2051
Epoch 2 Step 300 Loss: 7.5402
Epoch 2 Step 400 Loss: 7.8078
Epoch 2 Step 500 Loss: 8.1084
Epoch 2 Step 600 Loss: 7.6612
Epoch 2 Completed. Dev Perplexity: 2253.5049
Best model saved to TRANSFORMER_MH_pile.pt 

# Text Generation

This is your second main TODO. In the following cell, you will implement temperature sampling.

After you do this, you can run the following cell to generate from your language model. Don't worry if these outputs look a bit ungrammatical and weird. It shouldn't be entirely random tokens—but it will be close (at least with the default hyperparameters)!

Optionally, for extra credit, you can implement nucleus sampling in the subsequent cell. If you do this, be sure to fill out the written report telling us you did this, and giving some example generations.

In [9]:
import torch
import torch.nn as nn

def generate(model, tokenizer, start_text, max_new_tokens=30, device='cpu', temperature=0.8):
    model.eval()
    encoded = tokenizer.encode(start_text)
    tokens = torch.tensor(encoded, dtype=torch.long).unsqueeze(0).to(device)  # (1, T)

    eos_id = getattr(tokenizer, "eos_token_id", None)
    context_len = getattr(model, "context_len", None)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            model_input = tokens
            if context_len is not None and model_input.size(1) > context_len:
                model_input = model_input[:, -context_len:]  # keep only latest context

            output = model(model_input)  # (1, T, H)
            logits = torch.matmul(output, model.embedding.weight.t())  # (1, T, Vocab)

            # STUDENT START --------------------------
            # Get the logits for the last position only
            last_logits = logits[:, -1, :]  # (1, Vocab)
            # Apply temperature scaling
            scaled_logits = last_logits / temperature
            # Convert to probabilities
            probs = F.softmax(scaled_logits, dim=-1)  # (1, Vocab)
            # Sample from the distribution
            next_token = torch.multinomial(probs, num_samples=1)  # (1, 1)
            # STUDENT END ----------------------------

            tokens = torch.cat((tokens, next_token), dim=1)

            if eos_id is not None and next_token.item() == eos_id:
                break

    decoded = tokenizer.decode(tokens[0].tolist())
    return decoded

In [10]:
import torch
import torch.nn as nn

def nucleus_sampling_generate(model, tokenizer, start_text, max_new_tokens=30, device='cpu', p=0.9):
    model.eval()
    encoded = tokenizer.encode(start_text)
    tokens = torch.tensor(encoded, dtype=torch.long).unsqueeze(0).to(device)  # (1, T)

    eos_id = getattr(tokenizer, "eos_token_id", None)
    context_len = getattr(model, "context_len", None)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            model_input = tokens
            if context_len is not None and model_input.size(1) > context_len:
                model_input = model_input[:, -context_len:]  # keep only latest context

            output = model(model_input)  # (1, T, H)
            logits = torch.matmul(output, model.embedding.weight.t())  # (1, T, Vocab)

            # STUDENT START --------------------------
            # Get the logits for the last position only
            last_logits = logits[:, -1, :]  # (1, Vocab)
            # Convert to probabilities
            probs = F.softmax(last_logits, dim=-1)  # (1, Vocab)
            # Sort probabilities in descending order
            sorted_probs, sorted_indices = torch.sort(probs, descending=True)
            # Compute cumulative probabilities
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
            # Find the cutoff index where cumulative probability exceeds p
            sorted_indices_to_remove = cumulative_probs > p
            # Shift the mask right to keep the first token that exceeds the threshold
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = False
            # Zero out probabilities for tokens to remove
            sorted_probs[sorted_indices_to_remove] = 0.0
            # Renormalize
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            # Sample from the filtered distribution
            sampled_index = torch.multinomial(sorted_probs, num_samples=1)  # (1, 1)
            # Map back to original vocabulary indices
            next_token = sorted_indices.gather(-1, sampled_index)  # (1, 1)
            # STUDENT END ----------------------------

            tokens = torch.cat((tokens, next_token), dim=1)

            if eos_id is not None and next_token.item() == eos_id:
                break

    decoded = tokenizer.decode(tokens[0].tolist())
    return decoded

In [11]:
# Test generation with your trained model
prompts = [
    "The quick brown fox",
    "In the beginning",
    "What is 5 + 5?",
    "The meaning of life is"
]

print("Generating text samples with PILE model...\n")
for prompt in prompts:
    try:
        generated = generate(model_pile, tokenizer, prompt, max_new_tokens=30, device=device)
        print(f"Prompt: {prompt}")
        print(f"Generated: {generated}")
        print("-" * 80)
    except NotImplementedError:
        print("⚠️  Please implement the generate() function first!")
        break

Generating text samples with PILE model...

Prompt: The quick brown fox
Generated: The quick brown fox.d)- Tess A is of We data2, the the the surface by in marriage and it and the cats talking. But a a to
--------------------------------------------------------------------------------
Prompt: In the beginning
Generated: In the beginning is the mounting with the 1977 of refuted of once, mac is Mongo en).or:]. For findings.[[oly. The a evaluated generally those
--------------------------------------------------------------------------------
Prompt: What is 5 + 5?
Generated: What is 5 + 5?, and Sy and a a power. In the decline community +a, alternative-in. This's Facts. We exist (1 and E],[
--------------------------------------------------------------------------------
Prompt: The meaning of life is
Generated: The meaning of life is blog pathway the the single SUN- (;; Un Long injuries, the to 21 to be presence. In the long ATP is " Accordingly. Our any
----------------------------------

# Training and Fine-tuning on Shakespeare

In this section, you will train a series of Transformers on a small Shakespeare corpus. You will train a Transformer from scratch on this data, and compare this model to your pre-trained Pile model fine-tuned on the Shakespeare corpus. Finally, you will fine-tune GPT-2 (a model pre-trained on far more data than we have time for) on the Shakespeare corpus.

## Load Shakespeare Data

In [12]:
# Load Shakespeare dataset from HuggingFace
# https://huggingface.co/datasets/2nji/Shakespeare_Corpus/
print("Loading Shakespeare dataset...")
train_data_shakes = load_shakespeare_data('train')
dev_data_shakes = load_shakespeare_data('validation')

print(f"Shakespeare train data shape: {train_data_shakes.shape}")
print(f"Shakespeare dev data shape: {dev_data_shakes.shape}")

Loading Shakespeare dataset...
Loading Shakespeare train data from HuggingFace...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


README.md:   0%|          | 0.00/538 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/513k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/158k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/128k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4621 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1445 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1156 [00:00<?, ? examples/s]

Loaded 4621 Shakespeare train tokens
Loading Shakespeare validation data from HuggingFace...
Loaded 1156 Shakespeare validation tokens
Shakespeare train data shape: torch.Size([4621, 100])
Shakespeare dev data shape: torch.Size([1156, 100])


## Baseline: Evaluate Pile Model on Shakespeare (Zero-shot)

Before training from scratch, see how well your model pre-trained on the Pile does on Shakespeare without any adaptation.

In [13]:
pile_on_shakespeare_perp = evaluate_perplexity(model_pile, dev_data_shakes, device)
print(f"\Pile model on Shakespeare (zero-shot): {pile_on_shakespeare_perp:.4f}")
print("This is a baseline - fine-tuning should improve this!")

\Pile model on Shakespeare (zero-shot): 16.3631
This is a baseline - fine-tuning should improve this!


In [14]:
# Train from scratch
model_shakespeare = MultiHeadTransformer(
    vocab_size=vocab_size,
    hidden_dim=hidden_dim,
    context_len=context_len,
    num_heads=4
)
model_shakespeare.to(device)

print("Training model on Shakespeare...")

train(
    model_shakespeare,
    train_data_shakes,
    dev_data_shakes,
    epochs=10,
    lr=5e-3,
    device=device,
    save_path='TRANSFORMER_MH_shakespeare.pt'
)

# Evaluate
shakespeare_perp = evaluate_perplexity(model_shakespeare, dev_data_shakes, device)
print(f"\nDev perplexity: {shakespeare_perp:.4f}")

Training model on Shakespeare...
Starting training on mps...
Epoch 0 Step 0 Loss: 58.3752
Epoch 0 Step 100 Loss: 3.5332
Epoch 0 Step 200 Loss: 2.1147
Epoch 0 Completed. Dev Perplexity: 12.7133
Best model saved to TRANSFORMER_MH_shakespeare.pt with perplexity 12.7133
Epoch 1 Step 0 Loss: 1.8266
Epoch 1 Step 100 Loss: 2.4097
Epoch 1 Step 200 Loss: 3.0400
Epoch 1 Completed. Dev Perplexity: 11.1081
Best model saved to TRANSFORMER_MH_shakespeare.pt with perplexity 11.1081
Epoch 2 Step 0 Loss: 2.9636
Epoch 2 Step 100 Loss: 2.3258
Epoch 2 Step 200 Loss: 1.9122
Epoch 2 Completed. Dev Perplexity: 9.7219
Best model saved to TRANSFORMER_MH_shakespeare.pt with perplexity 9.7219
Epoch 3 Step 0 Loss: 1.7045
Epoch 3 Step 100 Loss: 1.6772
Epoch 3 Step 200 Loss: 2.4245
Epoch 3 Completed. Dev Perplexity: 9.2713
Best model saved to TRANSFORMER_MH_shakespeare.pt with perplexity 9.2713
Epoch 4 Step 0 Loss: 2.1929
Epoch 4 Step 100 Loss: 1.9628
Epoch 4 Step 200 Loss: 2.5762
Epoch 4 Completed. Dev Perplexity:

## Fine-tune on Shakespeare

In [15]:
# Load your pre-trained Pile model and fine-tune on Shakespeare
model_pile_shakespeare = MultiHeadTransformer(
    vocab_size=vocab_size,
    hidden_dim=hidden_dim,
    context_len=context_len,
    num_heads=4
)
model_pile_shakespeare.load_state_dict(torch.load('TRANSFORMER_MH_pile.pt', map_location=device))
model_pile_shakespeare.to(device)

print("Fine-tuning PILE model on Shakespeare...")

# Fine-tune on Shakespeare (use lower learning rates and num epochs for fine-tuning)
train(
    model_pile_shakespeare,
    train_data_shakes,
    dev_data_shakes,
    epochs=5,
    lr=5e-4,
    device=device,
    save_path='TRANSFORMER_MH_pile_shakespeare.pt'
)

# Evaluate
shakespeare_perp = evaluate_perplexity(model_pile_shakespeare, dev_data_shakes, device)
print(f"\nShakespeare-adapted model perplexity: {shakespeare_perp:.4f}")

Fine-tuning PILE model on Shakespeare...
Starting training on mps...
Epoch 0 Step 0 Loss: 3.0834
Epoch 0 Step 100 Loss: 1.6142
Epoch 0 Step 200 Loss: 2.4885
Epoch 0 Completed. Dev Perplexity: 8.6704
Best model saved to TRANSFORMER_MH_pile_shakespeare.pt with perplexity 8.6704
Epoch 1 Step 0 Loss: 2.3510
Epoch 1 Step 100 Loss: 2.2004
Epoch 1 Step 200 Loss: 1.6949
Epoch 1 Completed. Dev Perplexity: 8.0784
Best model saved to TRANSFORMER_MH_pile_shakespeare.pt with perplexity 8.0784
Epoch 2 Step 0 Loss: 2.6836
Epoch 2 Step 100 Loss: 2.2203
Epoch 2 Step 200 Loss: 2.3882
Epoch 2 Completed. Dev Perplexity: 7.8491
Best model saved to TRANSFORMER_MH_pile_shakespeare.pt with perplexity 7.8491
Epoch 3 Step 0 Loss: 2.4887
Epoch 3 Step 100 Loss: 1.7810
Epoch 3 Step 200 Loss: 2.2820
Epoch 3 Completed. Dev Perplexity: 7.6129
Best model saved to TRANSFORMER_MH_pile_shakespeare.pt with perplexity 7.6129
Epoch 4 Step 0 Loss: 2.1055
Epoch 4 Step 100 Loss: 2.9140
Epoch 4 Step 200 Loss: 2.5257
Epoch 4 Com

In [16]:
# Re-evaluate fine-tuned model's perplexity on the dev split of the Pile
# STUDENT START ----------------------------
pile_dev_loader = DataLoader(dev_data, batch_size=16)
finetuned_pile_perp = evaluate_perplexity(model_pile_shakespeare, pile_dev_loader, device)
print(f"Fine-tuned model perplexity on Pile dev: {finetuned_pile_perp:.4f}")
# STUDENT END ------------------------------

Fine-tuned model perplexity on Pile dev: 11083.8555


In [17]:
# Generate Shakespearean text
shakespeare_prompts = [
    "To be or not to be",
    "O Romeo, Romeo",
    "Friends, Romans, countrymen",
    "All the world's a stage"
]
prompts = [
    "The quick brown fox",
    "In the beginning",
    "What is 5 + 5?",
    "The meaning of life is"
]

model_pile_shakespeare.to(device)

for prompt in shakespeare_prompts:
    try:
        generated = generate(model_pile_shakespeare, tokenizer, prompt, max_new_tokens=50, device=device, temperature=0.9)
        print(f"Prompt: {prompt}")
        print(f"Generated: {generated}")
        print("-" * 80)
    except NotImplementedError:
        print("Please implement the generate() function first!")
        break

for prompt in prompts:
    try:
        generated = generate(model_pile_shakespeare, tokenizer, prompt, max_new_tokens=50, device=device, temperature=0.9)
        print(f"Prompt: {prompt}")
        print(f"Generated: {generated}")
        print("-" * 80)
    except NotImplementedError:
        print("Please implement the generate() function first!")
        break

Prompt: To be or not to be
Generated: To be or not to be you?<|endoftext|>
--------------------------------------------------------------------------------
Prompt: O Romeo, Romeo
Generated: O Romeo, Romeo: Merc: O's a fear:<|endoftext|>
--------------------------------------------------------------------------------
Prompt: Friends, Romans, countrymen
Generated: Friends, Romans, countrymen, an we be same he, that And at. I ambassadors to tim in talking are wife what thy.<|endoftext|>
--------------------------------------------------------------------------------
Prompt: All the world's a stage
Generated: All the world's a stage.<|endoftext|>
--------------------------------------------------------------------------------
Prompt: The quick brown fox
Generated: The quick brown fox: Upon that no word.'; be eyes.<|endoftext|>
--------------------------------------------------------------------------------
Prompt: In the beginning
Generated: In the beginning you.<|endoftext|>
-------------

# Fine-tune GPT-2 on Shakespeare

Compare your transformer with a pre-trained model. How much does the much larger scale of GPT-2's training data help?

## Fine-tune GPT-2

In [19]:
!pip install tf-keras -q

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [21]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset
import os

# Load GPT-2
MODEL_NAME = "gpt2"
print(f"Loading {MODEL_NAME}...")
tokenizer_hf = AutoTokenizer.from_pretrained(MODEL_NAME)
model_hf = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

if tokenizer_hf.pad_token is None:
    tokenizer_hf.pad_token = tokenizer_hf.eos_token

# Load Shakespeare dataset
print("Loading Shakespeare data...")
train_data_hf = load_shakespeare_data_hf('train')
dev_data_hf = load_shakespeare_data_hf('validation')

# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_list(train_data_hf)
dev_dataset = Dataset.from_list(dev_data_hf)

# Tokenize
def preprocess_function(examples):
    return tokenizer_hf(examples['text'], truncation=True, padding="max_length", max_length=128)

train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=['text'])
dev_dataset = dev_dataset.map(preprocess_function, batched=True, remove_columns=['text'])

# TODO: play around with these hyperparameters, and see how well you can
# optimize GPT-2's dev perplexity on Shakespeare data.
training_args = TrainingArguments(
    output_dir="./gpt2_shakespeare/",
    learning_rate=5e-4,     # Play around with this hyperparameter
    num_train_epochs=3,     # Play around with this hyperparameter
    per_device_train_batch_size=4,
    save_steps=500,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=100,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model_hf,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer_hf, mlm=False),
)

print("Starting GPT-2 fine-tuning...")
trainer.train()

print("Saving model...")
model_hf.save_pretrained("./gpt2_shakespeare/")
tokenizer_hf.save_pretrained("./gpt2_shakespeare/")
print("\nGPT-2 fine-tuning complete.")

Loading gpt2...
Loading Shakespeare data...
Loading Shakespeare train data for HuggingFace...
Loaded 4621 Shakespeare train lines
Loading Shakespeare validation data for HuggingFace...
Loaded 1156 Shakespeare validation lines


Map:   0%|          | 0/4621 [00:00<?, ? examples/s]

Map:   0%|          | 0/1156 [00:00<?, ? examples/s]

Starting GPT-2 fine-tuning...


/Users/tangzhengzheng/miniconda3/envs/AI001/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss
100,4.815500,4.662236
200,4.459100,4.541940
300,4.358000,4.399499
400,4.350800,4.349653
500,4.197300,4.314663
600,4.133000,4.266819
700,4.044800,4.238891
800,4.011500,4.190875
900,3.991600,4.174934
1000,3.947700,4.133238


/Users/tangzhengzheng/miniconda3/envs/AI001/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/tangzhengzheng/miniconda3/envs/AI001/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/tangzhengzheng/miniconda3/envs/AI001/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/tangzhengzheng/miniconda3/envs/AI001/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  

Saving model...

GPT-2 fine-tuning complete.


## Evaluate GPT-2

Compare GPT-2's perplexity with your custom transformer. These results may not be what you expect—but hold off your final judgment of quality until you see the generations.

In [22]:
eval_results = trainer.evaluate()
gpt2_loss = eval_results['eval_loss']
print(gpt2_loss)
gpt2_perplexity = torch.exp(torch.tensor(gpt2_loss)).item()

print(f"GPT-2 (Shakespeare fine-tuned):              {gpt2_perplexity:.4f}")

/Users/tangzhengzheng/miniconda3/envs/AI001/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


4.450124263763428
GPT-2 (Shakespeare fine-tuned):              85.6376


## Generate with GPT-2

In [23]:
# Generate Shakespeare with GPT-2
shakespeare_prompts = [
    "To be or not to be",
    "O Romeo, Romeo",
    "Friends, Romans, countrymen",
    "All the world's a stage"
]
prompts = [
    "The quick brown fox",
    "In the beginning",
    "What is 5 + 5?",
    "The meaning of life is"
]

model_hf.eval()
for prompt in shakespeare_prompts:
    inputs = tokenizer_hf(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model_hf.generate(
            inputs.input_ids,
            max_new_tokens=50,
            temperature=0.9,
            do_sample=True,
            pad_token_id=tokenizer_hf.eos_token_id
        )

    generated = tokenizer_hf.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Generated: {generated}")
    print("-" * 80)
for prompt in prompts:
    inputs = tokenizer_hf(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model_hf.generate(
            inputs.input_ids,
            max_new_tokens=50,
            temperature=0.9,
            do_sample=True,
            pad_token_id=tokenizer_hf.eos_token_id
        )

    generated = tokenizer_hf.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Generated: {generated}")
    print("-" * 80)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt: To be or not to be
Generated: To be or not to be. Therefore we'll issue out again, and again. What say you, sir? If any of your own choice, 'tis wit in request; For we have need of good cheer: 'tis fit for our purpose; My noble uncle is the
--------------------------------------------------------------------------------
Prompt: O Romeo, Romeo
Generated: O Romeo, Romeo, Romeo! thou hast no cause to fear. What comfort is this To thee that with the help of Mercutio? I am Angelo, and thyself preserve! If thou refuse, let me know thee will not; For, as thou hast
--------------------------------------------------------------------------------
Prompt: Friends, Romans, countrymen
Generated: Friends, Romans, countrymen, and all, and one amongst you. Fare you well. You know our enemy's house, our countrymen, Of whom you and yours are almost at odds; Neighbour Tybalt, our king, Is the house of Antium;
--------------------------------------------------------------------------------
Prompt: